In [ ]:

import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# 1. Create the student dataset
df = pd.DataFrame({
    "Study": ["High", "High", "High", "Low", "Low", "Low", "Low", "High"],
    "Attendance": ["Good", "Good", "Poor", "Good", "Poor", "Poor", "Good", "Poor"],
    "Homework": ["Done", "Missed", "Done", "Done", "Missed", "Done", "Missed", "Done"],
    "Result": ["Pass", "Pass", "Pass", "Pass", "Fail", "Fail", "Fail", "Pass"]
})

# 2. Separate features and target
X = pd.get_dummies(df.drop(columns="Result"), dtype=int)
y = df["Result"]

In [ ]:
# 3. Train a decision tree
tree = DecisionTreeClassifier(
    criterion="entropy", max_depth=3, random_state=42
)
tree.fit(X, y)

DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)

In [ ]:
# 4. Show the learned rules
print("Decision Tree Rules:")
print(export_text(tree, feature_names=list(X.columns)))

Decision Tree Rules:
|--- Study_High <= 0.50
|   |--- Attendance_Poor <= 0.50
|   |   |--- Homework_Done <= 0.50
|   |   |   |--- class: Fail
|   |   |--- Homework_Done >  0.50
|   |   |   |--- class: Pass
|   |--- Attendance_Poor >  0.50
|   |   |--- class: Fail
|--- Study_High >  0.50
|   |--- class: Pass



In [ ]:
# 5. Predict a new student
new_student = pd.DataFrame([{
    "Study": "Low",
    "Attendance": "Good",
    "Homework": "Done"
}])

new_student = pd.get_dummies(new_student, dtype=int)
new_student = new_student.reindex(columns=X.columns, fill_value=0)

print("Tree Prediction:", tree.predict(new_student)[0])
print("Training Accuracy:", tree.score(X, y))

Tree Prediction: Pass
Training Accuracy: 1.0


In [ ]:
# 6. Compare Entropy and Gini
for method in ["entropy", "gini"]:
    model = DecisionTreeClassifier(
        criterion=method, max_depth=3, random_state=42
    )
    model.fit(X, y)
    print(method, "Training Accuracy:", model.score(X, y))

entropy Training Accuracy: 1.0
gini Training Accuracy: 1.0


In [ ]:
# 7. Compare tree depths (overfitting demonstration)
for depth in [1, 2, 3, None]:
    model = DecisionTreeClassifier(
        max_depth=depth, random_state=42
    )
    model.fit(X, y)
    print("Depth:", depth, "Accuracy:", model.score(X, y))

Depth: 1 Accuracy: 0.875
Depth: 2 Accuracy: 0.875
Depth: 3 Accuracy: 1.0
Depth: None Accuracy: 1.0


In [ ]:
# 8. Train a random forest
forest = RandomForestClassifier(
    n_estimators=100, max_features="sqrt",
    random_state=42
)
forest.fit(X, y)

print("Random Forest Prediction:", forest.predict(new_student)[0])
print("Forest Training Accuracy:", forest.score(X, y))

Random Forest Prediction: Pass
Forest Training Accuracy: 1.0


In [ ]:
# 9. Show feature importance
importance = pd.Series(
    forest.feature_importances_, index=X.columns
).sort_values(ascending=False)

print("\nFeature Importance:")
print(importance)


Feature Importance:
Study_High         0.278792
Study_Low          0.252289
Attendance_Good    0.130538
Homework_Done      0.119535
Homework_Missed    0.112194
Attendance_Poor    0.106652
dtype: float64
